# D1 · Reading the training curves of XGBoost

**Type**: Interpret · **Level**: ●●○ independent · **Time**: about 50 minutes · **core path ★**

## Objective

Read `evals_result()` of four configurations (depth 2, 4, 6 and 6 with `min_child_weight=20`) trained for 600 rounds without early stopping: where the validation log-loss minimum is, how large the training-validation gap is at the minimum and at the end, and which configuration is the best and why. Then
predict the early stopping round by looking at the curve.

## Dataset

`make_classification(4000, 12 features, 5 informative, flip_y=0.08)`, 60% training, 20% validation, 20% test split. Models: `XGBClassifier(n_estimators=600, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, eval_metric="logloss")`, `evals_result` already computed in `curves[name]` with keys `validation_0` (training) and `validation_1` (validation).

## Educational purpose

Learn to read the two curves and recognize what they say: the validation minimum is the "right round"; the gap with the training curve is a measure of overfitting; depth and `min_child_weight` change both the minimum and the gap.

## Task

1. For each configuration (`curves` is a dictionary name → result of `evals_result()`), build `table` (index: configurations; columns `min_round` (1-based index of the validation minimum), `val_min` (minimum value), `train_at_min`, `gap_at_min` (`val_min − train_at_min`), `final_val` and `final_gap`).
2. Save `best` (configuration with the lowest `val_min`) and `most_overfit` (configuration with the largest `final_gap`).
3. Save `degradation` (`pd.Series`, `final_val − val_min` per configuration): how much the validation gets worse if you do not stop at the minimum.
4. Simulate early stopping with patience 30: `es_round` (dictionary configuration → first round `t`, from 1, such that the minimum of `val[:t]` was reached 30 rounds earlier, that is `argmin(val[:t]) + 1 + 30 == t`) and `es_cost` (`pd.Series`: the `val_min` you would find by stopping at `es_round`, that is `min(val[:es_round])`, minus the `val_min` of the table). Note that the stop triggers at the **first** plateau: it can come before the global minimum.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import xgboost as xgb
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(4000, n_features=12, n_informative=5, class_sep=0.8, flip_y=0.08, random_state=0)
X_tr, X_rest, y_tr, y_rest = train_test_split(X, y, test_size=0.4, random_state=0)
X_val, X_te, y_val, y_te = train_test_split(X_rest, y_rest, test_size=0.5, random_state=0)
configs = {"depth 2": dict(max_depth=2), "depth 4": dict(max_depth=4), "depth 6": dict(max_depth=6), "depth 6, min_child_weight 20": dict(max_depth=6, min_child_weight=20)}
curves = {}
for name, kw in configs.items():
    m = xgb.XGBClassifier(n_estimators=600, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", n_jobs=-1, random_state=0, **kw)
    m.fit(X_tr, y_tr, eval_set=[(X_tr, y_tr), (X_val, y_val)], verbose=False)
    curves[name] = m.evals_result()
print(list(curves))

## Your work

In [ ]:
table = pd.DataFrame(index=list(curves), columns=["min_round", "val_min", "train_at_min", "gap_at_min", "final_val", "final_gap"], dtype=float)
for name, r in curves.items():
    train, val = np.array(r["validation_0"]["logloss"]), np.array(r["validation_1"]["logloss"])
    ...
table.round(3)

In [ ]:
best = None
most_overfit = None
degradation = None
es_round = {}
es_cost = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`min_round = int(val.argmin()) + 1`; `train_at_min = train[val.argmin()]`. Early stopping: for `t` from 1 to 600 the first `t` for which `val[:t].argmin() + 1 + 30 == t`; then `val[:t].min() - val.min()`.'):
    check(table.notna().all().all() and list(table.columns) == ["min_round", "val_min", "train_at_min", "gap_at_min", "final_val", "final_gap"], "complete table", "one row per configuration")
    check(table["min_round"].is_monotonic_decreasing is False and table.loc["depth 2", "min_round"] > table.loc["depth 6", "min_round"] + 100, "shallow trees reach the minimum much later (over 100 rounds after the deep ones)", "min_round")
    check(best in table.index and table.loc[best, "val_min"] == table["val_min"].min() and table["val_min"].max() - table["val_min"].min() < 0.05, "the four configurations have a similar validation minimum (difference below 0.05): depth matters less than the right round", "val_min")
    check(most_overfit == "depth 6" and table.loc["depth 6", "final_gap"] > 2 * table.loc["depth 2", "final_gap"], "depth 6 without constraints has the largest final gap (over double that of depth 2)", "final_gap")
    check(degradation["depth 6"] > 0.03 and degradation["depth 2"] < 0.01 and degradation["depth 6, min_child_weight 20"] < degradation["depth 6"] / 3, "without early stopping depth 6 loses over 0.03 of log-loss; depth 2 less than 0.01; min_child_weight cuts the effect to less than a third", "final_val - val_min")
    check(len(es_round) == 4 and all(30 < es_round[n] <= table.loc[n, "min_round"] + 30 for n in table.index), "the stop triggers at the first plateau: never after the global minimum plus 30 rounds, often before", "argmin(val[:t]) + 1 + 30 == t")
    check(es_cost is not None and (es_cost > -1e-9).all() and (es_cost < 0.01).all(), "stopping at the first plateau costs less than 0.01 of validation log-loss compared with the global minimum", "min(val[:es_round]) - val_min")
done()

---
The commented solution is in `solutions/D_interpret/D1_reading_the_training_curves.ipynb`. Open it after you pass the check, or after you have really tried.